# Stage 3 v2 — Semantic Chunking with NS Relevance Filtering

Improvements over v1:
- **All documents** go through sentence splitting + similarity grouping — no more fast-path
  shortcut that left mixed-topic short docs as a single chunk.
- **NS chunk filter integrated**: only NS-relevant chunks emitted. No separate `ns_filter.py` step.
- **Two-pass batch embedding**: sentences embedded (GPU, batch 512) for grouping;
  final chunks re-embedded with context prefix (GPU, batch 64).
- Trailing single-sentence fragments are merged into the previous chunk (`CHUNK_MIN_SENTENCES` rule).

**Datasets needed:**
- `ns-sentiment-clean` — `submissions_clean.parquet`, `comments_clean.parquet` (Stage 2 outputs)

**Setup:** GPU T4 x2 · Save & Run All (committed mode)

In [ ]:
# Cell 1 — Discover input paths
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 2 — Config  ← UPDATE PATHS AFTER RUNNING CELL 1
SUBMISSIONS_CLEAN = '/kaggle/input/<your-dataset>/submissions_clean.parquet'
COMMENTS_CLEAN    = '/kaggle/input/<your-dataset>/comments_clean.parquet'
OUT_DIR           = '/kaggle/working'

EMBEDDING_MODEL      = 'sentence-transformers/all-mpnet-base-v2'
CHUNK_MIN_SENTENCES  = 2
CHUNK_MAX_SENTENCES  = 6
SIMILARITY_THRESHOLD = 0.5
DEVICE               = 'cuda'

_NS_SUBREDDIT = 'nationalservicesg'

In [ ]:
# Cell 3 — Install + imports
!pip install spacy sentence-transformers -q

import re
import numpy as np
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('Imports OK')

In [ ]:
# Cell 4 — NS relevance filter patterns
_STRONG_TERMS = {
    'enlistment', 'enlist', 'enlisted', 'enlistee',
    'reservist', 'reservists',
    'ord', 'rod',
    'ippt',
    'ict',
    'bmt', 'bmtc', 'tekong',
    'ocs', 'scs',
    'scdf', 'spf', 'ndu',
    'conscription', 'conscript', 'nsman', 'nsmen', 'ns man',
    'national service',
    'mindef',
    'pes status', 'downpes', 'medical board',
    'guard duty', 'book in', 'book out', 'confined to camp',
    'route march', 'outfield',
    'chao keng', 'sign extra',
    'emart',
}
_WEAK_TERMS = {
    'ns', 'nsf', 'saf', 'bmt', 'pes', 'db', 'recruit',
    'sergeant', 'encik', 'vocation', 'wayang', 'ippt',
    'reservist', 'ict', 'rsi', 'rso', 'officer cadet',
    'in-camp', 'ns training', 'operationally ready',
    'ns deferment', 'ns disruption',
    'defend singapore', 'serve nation',
    'tekkan', 'arrowed', 'siao on',
    'off day', 'leave pass',
}

_strong_pat = re.compile(
    '|'.join(re.escape(t) for t in sorted(_STRONG_TERMS, key=len, reverse=True)),
    re.IGNORECASE,
)
_weak_pat = re.compile(
    '|'.join(r'\b' + re.escape(t) + r'\b' for t in sorted(_WEAK_TERMS, key=len, reverse=True)),
    re.IGNORECASE,
)

def _is_ns_chunk(text):
    if _strong_pat.search(text):
        return True
    return len({m.lower() for m in _weak_pat.findall(text)}) >= 2

print('NS filter patterns compiled')

In [ ]:
# Cell 5 — Load models
nlp   = spacy.blank('en')
nlp.add_pipe('sentencizer')

model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print(f'Embedding model loaded on {DEVICE}')

In [ ]:
# Cell 6 — Core chunking functions

import torch

def split_sentences(texts):
    result = []
    for doc in nlp.pipe(texts, batch_size=512):
        sents = [s.text.strip() for s in doc.sents if s.text.strip()]
        result.append(sents if sents else [''])
    return result


def group_into_chunks(sentences, sent_embeddings):
    if len(sentences) == 1:
        return [sentences[0]]

    groups = []
    current = [sentences[0]]

    for i in range(1, len(sentences)):
        sim = cosine_similarity(
            sent_embeddings[i - 1].reshape(1, -1),
            sent_embeddings[i].reshape(1, -1),
        )[0][0]
        over_max       = len(current) >= CHUNK_MAX_SENTENCES
        semantic_break = sim < SIMILARITY_THRESHOLD and len(current) >= CHUNK_MIN_SENTENCES

        if over_max or semantic_break:
            groups.append(current)
            current = [sentences[i]]
        else:
            current.append(sentences[i])

    if len(current) < CHUNK_MIN_SENTENCES and groups:
        groups[-1].extend(current)
    else:
        groups.append(current)

    return [' '.join(g).strip() for g in groups if ' '.join(g).strip()]


def _process_doc_batch(batch_rows, batch_sentences, batch_sent_embs, title_map, metadata_cols, doc_type):
    """Group + NS filter one batch of documents. Returns list of record dicts."""
    records = []
    n_filtered = 0
    for row, sents, sent_embs in zip(batch_rows, batch_sentences, batch_sent_embs):
        subreddit = str(getattr(row, 'subreddit', '')).lower()
        doc_id    = str(row.id)
        is_ns_sr  = subreddit == _NS_SUBREDDIT

        chunk_texts = group_into_chunks(sents, sent_embs)

        prefix = ''
        if title_map is not None and hasattr(row, 'post_id'):
            title  = title_map.get(str(row.post_id), '')
            prefix = f'Post: {title}\n\nComment: ' if title else ''

        meta = {c: getattr(row, c) for c in metadata_cols if hasattr(row, c)}
        kept = 0
        for ct in chunk_texts:
            if not is_ns_sr and not _is_ns_chunk(ct):
                n_filtered += 1
                continue
            records.append({
                'doc_id':   doc_id,
                'doc_type': doc_type,
                'text':     ct,
                '_embed':   f'{prefix}{ct}' if prefix else ct,
                '_seq':     kept,
                **meta,
            })
            kept += 1

        if kept > 0:
            for r in records[-kept:]:
                r['_total'] = kept

    return records, n_filtered


def chunk_dataframe(df, text_col, doc_type, metadata_cols, title_map=None, doc_batch_size=5000):
    """
    Four-phase chunker with document-level batching to avoid GPU OOM.

    Phase 1 — Sentence split all documents (spaCy, CPU).
    Phase 2 — Embed sentences in doc-batches (GPU, batch_size=128).
    Phase 3 — Group into chunks + NS filter per batch.
    Phase 4 — Batch embed all surviving chunks with context prefix.
    """
    texts = df[text_col].fillna('').tolist()
    rows  = list(df.itertuples(index=False))

    # Phase 1: sentence splitting (full corpus, CPU)
    print(f'  Phase 1: splitting {len(texts):,} documents into sentences …')
    all_sentences = split_sentences(texts)
    print(f'  Total sentences: {sum(len(s) for s in all_sentences):,}')

    # Phases 2 + 3: process in doc-batches to cap GPU memory
    print(f'  Phases 2+3: embedding sentences + grouping (doc batches of {doc_batch_size:,}) …')
    all_records  = []
    total_filtered = 0
    n_batches = (len(rows) + doc_batch_size - 1) // doc_batch_size

    for b in range(n_batches):
        lo = b * doc_batch_size
        hi = min(lo + doc_batch_size, len(rows))
        batch_rows  = rows[lo:hi]
        batch_sents = all_sentences[lo:hi]

        flat_sents = [s for sents in batch_sents for s in sents]
        flat_embs  = model.encode(
            flat_sents, batch_size=128, show_progress_bar=False, convert_to_numpy=True,
        )
        torch.cuda.empty_cache()

        # Reconstruct per-doc embeddings for this batch
        batch_embs = []
        idx = 0
        for sents in batch_sents:
            n = len(sents)
            batch_embs.append(flat_embs[idx: idx + n])
            idx += n
        del flat_embs

        batch_records, n_filt = _process_doc_batch(
            batch_rows, batch_sents, batch_embs, title_map, metadata_cols, doc_type,
        )
        all_records.extend(batch_records)
        total_filtered += n_filt

        print(f'    Batch {b + 1}/{n_batches}  docs {lo:,}–{hi:,}  '
              f'chunks so far: {len(all_records):,}')

    print(f'  NS filter removed {total_filtered:,} off-topic chunks')
    print(f'  Surviving chunks: {len(all_records):,}')

    if not all_records:
        return pd.DataFrame()

    for r in all_records:
        total        = r.pop('_total', 1)
        seq          = r.pop('_seq')
        r['chunk_idx']   = seq
        r['chunk_count'] = total
        r['chunk_id']    = f"{r['doc_id']}_{seq}"

    # Phase 4: batch embed all chunks with context prefix
    print('  Phase 4: batch embedding chunks with context prefix …')
    embed_texts = [r.pop('_embed') for r in all_records]
    chunk_embs  = model.encode(
        embed_texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True,
    )
    for r, emb in zip(all_records, chunk_embs):
        r['embedding'] = emb.tolist()

    return pd.DataFrame(all_records)

print('Functions defined')

In [ ]:
# Cell 7 — Chunk submissions
subs = pd.read_parquet(SUBMISSIONS_CLEAN)
print(f'Submissions loaded: {len(subs):,}')
print(f'Columns: {subs.columns.tolist()}')
print(f'Subreddit breakdown:\n{subs["subreddit"].value_counts().to_string()}')

# title may not be present if it was dropped after being merged into combined_text
if 'title' in subs.columns:
    title_map = dict(zip(subs['id'].astype(str), subs['title'].astype(str)))
else:
    # Fall back to using combined_text prefix up to the first sentence
    title_map = {}
    print('Warning: title column not found — comment context prefixing disabled')
print(f'Title map: {len(title_map):,} entries')

sub_meta = [
    'subreddit', 'created_utc', 'score', 'log_weight',
    'upvote_ratio', 'num_comments', 'text_source', 'author', 'permalink',
]
sub_chunks = chunk_dataframe(subs, 'combined_text', 'submission', sub_meta)
sub_chunks.to_parquet(f'{OUT_DIR}/submissions_chunks.parquet', index=False)
print(f'Saved submissions_chunks.parquet  ({len(sub_chunks):,} chunks)')
del subs, sub_chunks

In [ ]:
# Cell 8 — Chunk comments
coms = pd.read_parquet(COMMENTS_CLEAN)
print(f'Comments loaded: {len(coms):,}')
print(f'Subreddit breakdown:\n{coms["subreddit"].value_counts().to_string()}')

com_meta = [
    'subreddit', 'created_utc', 'score', 'log_weight',
    'author', 'post_id', 'permalink', 'parent_id', 'depth',
]
com_chunks = chunk_dataframe(coms, 'body', 'comment', com_meta, title_map=title_map)
com_chunks.to_parquet(f'{OUT_DIR}/comments_chunks.parquet', index=False)
print(f'Saved comments_chunks.parquet  ({len(com_chunks):,} chunks)')
del coms, com_chunks

In [ ]:
# Cell 9 — Quality summary
sub_chunks = pd.read_parquet(f'{OUT_DIR}/submissions_chunks.parquet')
com_chunks = pd.read_parquet(f'{OUT_DIR}/comments_chunks.parquet')

print('── Submissions ──')
print(f'  Total chunks:    {len(sub_chunks):,}')
print(f'  Subreddit:\n{sub_chunks["subreddit"].value_counts().to_string()}')
print(f'  Chunks per doc:  {sub_chunks["chunk_count"].describe().to_string()}')
print(f'  Char length:     {sub_chunks["text"].str.len().describe().to_string()}')

print('\n── Comments ──')
print(f'  Total chunks:    {len(com_chunks):,}')
print(f'  Subreddit:\n{com_chunks["subreddit"].value_counts().to_string()}')
print(f'  Chunks per doc:  {com_chunks["chunk_count"].describe().to_string()}')
print(f'  Char length:     {com_chunks["text"].str.len().describe().to_string()}')

# Flag any chunks still under 50 chars (sanity check)
all_chunks = pd.concat([sub_chunks, com_chunks], ignore_index=True)
tiny = (all_chunks['text'].str.len() < 50).sum()
print(f'\nChunks under 50 chars (sanity): {tiny:,}')
print('\nStage 3 v2 complete.')